# Proyecto de Primer Bimestre  
## Sistema de Recuperación de Información  

**Integrantes:** Bautista Alexis - Correa Francisco  
**Fecha de entrega:** 1 de junio de 2026

In [1]:
%load_ext autoreload
%autoreload 2

### a. Construcción del índice

In [ ]:
import preprocesamiento

Leer un corpus de documentos en texto plano.

In [3]:
corpus = preprocesamiento.cargar_corpus()

Se encontraron 8 archivos CSV en el directorio.
Carga completa. El corpus tiene un total de 51081 documentos individuales.


Procesamiento básico: tokenización, normalización y remoción de stopwords.

In [4]:
corpus_procesado = []
for doc in corpus:
    tokens_doc = preprocesamiento.preprocesar(doc)
    corpus_procesado.append(tokens_doc)

print(f"Se han preprocesado {len(corpus_procesado)} documentos.")

Se han preprocesado 51081 documentos.


Construcción de un índice invertido que almacene, para cada término, los documentos en los que
aparece y su frecuencia.

In [5]:
import indice_invertido

In [6]:
indice = indice_invertido.crear_indice_invertido(corpus_procesado)

El indice a sido creado correctamente


In [7]:
# prueba rapida
if "bank" in indice:
    for doc_id, freq in list(indice["bank"].items())[:5]:
        print(f"{doc_id}: {freq}")
else:
    print("El término no existe en el corpus")

4: 1
9: 15
12: 5
13: 2
14: 3


**Ejemplos de interpretacion:**

4: 1 -> En el documento con el ID 4, la palabra "bank" aparece 1 vez.  
9: 15 -> En el documento con el ID 9, la palabra "bank" se repite 15 veces.  
12: 5 -> En el documento con el ID 12, la palabra "bank" se encuentra 5 veces.

In [8]:
# prueba rapida
print(indice.get("DDada", "El término no existe en el corpus"))

El término no existe en el corpus


### b. Modelo de recuperación

Implementar recuperación basada en similitud Jaccard utilizando vectores binarios

In [9]:
import modelos

In [10]:
query = input ("Ingrese una consulta: ")

In [11]:
ranking = modelos.recuperar_jaccard(query, corpus_procesado)

print(f"Resultados para: '{query}'")
for doc_id, score in ranking[:5]: # Mostrar el top 5
    
    print(f"Documento ID: {doc_id} | Similitud: {score * 100:.2f}%")

Resultados para: 'dog'
Documento ID: 22060 | Similitud: 5.00%
Documento ID: 46599 | Similitud: 5.00%
Documento ID: 22064 | Similitud: 4.00%
Documento ID: 46603 | Similitud: 4.00%
Documento ID: 30416 | Similitud: 3.23%


Implementar recuperación basada en similitud de coseno utilizando TF-IDF

In [12]:
ranking_tfidf = modelos.recuperar_tfidf(query, corpus_procesado)

print(f"Resultados TF-IDF para: '{query}'")
for doc_id, score in ranking_tfidf[:5]: # Mostrar el top 5
    # similitud coseno entre 0 y 1
    print(f"Documento ID: {doc_id} | Similitud Coseno: {score:.4f}")

Resultados TF-IDF para: 'dog'
Documento ID: 7479 | Similitud Coseno: 0.3123
Documento ID: 19641 | Similitud Coseno: 0.3123
Documento ID: 44183 | Similitud Coseno: 0.3123
Documento ID: 22060 | Similitud Coseno: 0.2475
Documento ID: 46599 | Similitud Coseno: 0.2475


Implementar recuperación con BM25.

In [13]:
ranking_bm25 = modelos.recuperar_bm25(query, corpus_procesado, indice)

print(f"Resultados BM25 para: '{query}'")
for doc_id, score in ranking_bm25[:5]:
    print(f"Documento ID: {doc_id} | Score BM25: {score:.4f}")

Resultados BM25 para: 'dog'
Documento ID: 7479 | Score BM25: 12.5991
Documento ID: 19641 | Score BM25: 12.5991
Documento ID: 44183 | Score BM25: 12.5991
Documento ID: 22060 | Score BM25: 10.9901
Documento ID: 46599 | Score BM25: 10.9901


**Interpretación de Métricas BM25**

* **Scores no normalizados:** Los valores obtenidos (ej. 7.1117) no son porcentajes ni probabilidades (0-100%). Son métricas relativas que solo sirven para comparar qué documento es más relevante que otro dentro de la *misma* consulta.
* **Empates matemáticos:** Los scores idénticos ocurren cuando los documentos son textos duplicados, o cuando coinciden exactamente en su longitud total y en la cantidad de veces que repiten los términos buscados.
* **Criterios de relevancia:** Un score alto indica que el documento contiene las palabras más "raras" de la búsqueda (alto IDF), las menciona de forma natural sin hacer spam (saturación de TF) y es un texto relativamente conciso (penalización a documentos muy largos).

### c. Interfaz básica

### d. Recuperación semántica con embeddings

• Generar embeddings para los documentos del corpus utilizando un modelo preentrenado.  
• Generar embeddings para las consultas de texto libre.  
• Almacenar los embeddings en una base de datos vectorial, como ChromaDB o FAISS.  
• Recuperar los documentos más similares usando búsqueda vectorial.  
• Mostrar un ranking de resultados basado en similitud vectorial.  

Para modelos clasicos como TF-IDF, BM25 es necesrio tener los tokens limpios y el stemming (ej. ["japan", "bank", "tax"]). Sin embargo, a los modelos semánticos (Transformers) les hace daño el preprocesamiento agresivo. Estos modelos necesitan leer el texto con su sintaxis, puntuación y conectores (stop words) para entender el contexto real de la oración.

Por esto se usara el corpus original para esta seccion. Ademas se decidio usar la base de datos vectorial FAISS. Por ultimo se decidio usar el modelo all-MiniLM-L6-v2 ya que es el estándar de la industria para este tipo de proyectos académicos porque es extremadamente rápido, pesa poco y ofrece una precisión altísima para representar oraciones en inglés.

In [ ]:
import modelo_semantico

modelo_transformer, base_vectorial_faiss = modelo_semantico.construir_indice_faiss(corpus)

In [15]:
#prueba
mi_busqueda = "cat in blue house"

ranking_semantico = modelo_semantico.recuperar_semantico(
    query_texto=mi_busqueda, 
    modelo=modelo_transformer, 
    indice_faiss=base_vectorial_faiss, 
    top_k=5
)

print(f"Resultados Semánticos para: '{mi_busqueda}'")
for doc_id, score in ranking_semantico:
    print(f"Documento ID: {doc_id} | Similitud Semántica: {score:.4f}")

Resultados Semánticos para: 'cat in blue house'
Documento ID: 5376 | Similitud Semántica: 0.3959
Documento ID: 16753 | Similitud Semántica: 0.3959
Documento ID: 41297 | Similitud Semántica: 0.3959
Documento ID: 26401 | Similitud Semántica: 0.3903
Documento ID: 50174 | Similitud Semántica: 0.3903


### e. Evaluación de resultados

En esta sección se usa únicamente el conjunto de prueba para construir una comparación entre Jaccard, TF-IDF, BM25 y recuperación semántica.

Como este corpus no trae consultas separadas ni un archivo de qrels externo, se usa la columna `topics` como criterio de relevancia: un documento es relevante para una consulta si contiene ese tema.

In [19]:
import evaluacion
import pandas as pd
import os

# 1. Cargamos el DataFrame completo
archivos = sorted([f for f in os.listdir("corpus") if f.endswith(".csv")])
df_corpus = pd.concat([pd.read_csv(f"corpus/{f}", encoding="utf-8") for f in archivos], ignore_index=True)

#  Eliminamos textos nulos y reseteamos el ID 
df_corpus = df_corpus.dropna(subset=['text']).reset_index(drop=True)

consultas_prueba = ["earn", "acq", "crude", "trade", "money-fx", "grain", "cocoa"]

# 2. Generamos el diccionario de qrels
qrels_eval = evaluacion.construir_qrels_desde_dataframe(df_corpus, consultas_prueba)
print(f"Qrels generados para {len(qrels_eval)} consultas.")

Qrels generados para 7 consultas.


In [20]:
# 1. Diccionario para guardar los resultados de todos los modelos
resultados_globales = {"jaccard": {}, "tfidf": {}, "bm25": {}, "semantico": {}}

# 2. Ejecutamos cada consulta en todos los modelos (Usamos las variables que YA tienes en tu notebook)
print("Ejecutando consultas de prueba...")
for query in consultas_prueba:
    resultados_globales["jaccard"][query] = modelos.recuperar_jaccard(query, corpus_procesado)
    resultados_globales["tfidf"][query] = modelos.recuperar_tfidf(query, corpus_procesado)
    resultados_globales["bm25"][query] = modelos.recuperar_bm25(query, corpus_procesado, indice)
    
    # Para el semántico usamos el texto original y el índice de FAISS
    resultados_globales["semantico"][query] = modelo_semantico.recuperar_semantico(
        query, modelo_transformer, base_vectorial_faiss, top_k=20
    )

# 3. Evaluamos y comparamos (k=10 por defecto)
df_comparacion = evaluacion.comparar_modelos(resultados_globales, qrels_eval, k=10)

# Mostrar la tabla final en el notebook
print("\n--- COMPARACIÓN FINAL DE MODELOS ---")
df_comparacion

Ejecutando consultas de prueba...

--- COMPARACIÓN FINAL DE MODELOS ---


,Modelo,MAP@10,Precision Media@10,Recall Medio@10
0,TFIDF,0.009780,0.385714,0.012626
1,BM25,0.007458,0.414286,0.009903
2,SEMANTICO,0.002923,0.385714,0.007970
3,JACCARD,0.000724,0.200000,0.000898


**Precision Media@10:** Nos dice qué porcentaje de los 10 documentos recuperados eran realmente útiles.

**Recall Medio@10:** Evalúa cuántos documentos relevantes del total logramos rescatar.

**MAP@10 (Mean Average Precision):** Revisa si los pocos aciertos que tuvimos aparecieron en el primer puesto o en el último.

In [24]:
#Prueba con una oracion (no un solo termino)
# 1. Definimos una consulta con contexto y lenguaje natural
consulta_contexto = "reports on company earnings, quarterly profits and banking sector"

print(f"Buscando: '{consulta_contexto}'\n")

# --- RECUPERACIÓN LÉXICA (BM25) ---
print("Top 3 - Modelo BM25 (coincidencias exactas de palabras):")
ranking_bm25_ctx = modelos.recuperar_bm25(consulta_contexto, corpus_procesado, indice)

for doc_id, score in ranking_bm25_ctx[:3]:
    # Extraemos los primeros 200 caracteres del texto real
    texto_real = str(df_corpus['text'].iloc[doc_id])[:200].replace('\n', ' ')
    print(f"➤ ID {doc_id} | Score: {score:.2f}")
    print(f"  Snippet: {texto_real}...\n")


print("-" * 70 + "\n")


# --- RECUPERACIÓN SEMÁNTICA (TRANSFORMER) ---
print("Top 3 - Modelo Semántico:")
ranking_sem_ctx = modelo_semantico.recuperar_semantico(
    consulta_contexto, modelo_transformer, base_vectorial_faiss, top_k=3
)

for doc_id, score in ranking_sem_ctx:
    texto_real = str(df_corpus['text'].iloc[doc_id])[:200].replace('\n', ' ')
    print(f"➤ ID {doc_id} | Similitud: {score:.4f}")
    print(f"  Snippet: {texto_real}...\n")

Buscando: 'reports on company earnings, quarterly profits and banking sector'

Top 3 - Modelo BM25 (coincidencias exactas de palabras):
➤ ID 3726 | Score: 16.11
  Snippet: &lt;Royal Bank of Canada>, in reporting a 19 pct drop in first quarter earnings, said it expects to report improved results in future earnings periods.     "Healthy consumer credit growth, record fee-...

➤ ID 14402 | Score: 16.11
  Snippet: &lt;Royal Bank of Canada>, in reporting a 19 pct drop in first quarter earnings, said it expects to report improved results in future earnings periods.     "Healthy consumer credit growth, record fee-...

➤ ID 38946 | Score: 16.11
  Snippet: &lt;Royal Bank of Canada>, in reporting a 19 pct drop in first quarter earnings, said it expects to report improved results in future earnings periods.     "Healthy consumer credit growth, record fee-...

----------------------------------------------------------------------

Top 3 - Modelo Semántico:
➤ ID 17559 | Similitud: 0.5854
  Snippet:

### f. Comparación de Modelos y Análisis de Resultados

En base a la evaluación cuantitativa utilizando Precision@10, Recall@10 y MAP@10 sobre el corpus ModApte, se observan las siguientes tendencias:

1. **Modelos Léxicos :** `BM25` y `TF-IDF` demostraron ser superiores para este tipo de dataset. `BM25` obtuvo la mejor Precisión Media (~41.4%), demostrando su robustez al penalizar documentos largos. Sin embargo, `TF-IDF` obtuvo un `MAP@10` ligeramente superior, indicando que posicionó los aciertos en los escalones más altos del ranking (Top 1-3).
2. **El límite del Modelo Binario:** La similitud de `Jaccard` quedó rezagada con la menor precisión (20%). Al no considerar la frecuencia de los términos (TF), pierde la capacidad de distinguir entre un documento fuertemente enfocado en el tema y uno que solo menciona el término tangencialmente.
3. **Comportamiento de la Recuperación Semántica:** * **¿Cuándo empeora?** El modelo basado en embeddings tuvo un rendimiento inferior a `BM25` y `TF-IDF` en esta prueba. Esto se debe a la naturaleza de las consultas (short-queries de una sola palabra clave como *"earn"* o *"crude"*). Los modelos semánticos sufren con entradas sin contexto, ya que intentan proyectar significados abstractos en lugar de hacer coincidencias léxicas exactas.
   * **¿Cuándo funciona mejor?** La recuperación semántica superaría a `BM25` en consultas de lenguaje natural descriptivo (ej. *"¿Cuáles son las ganancias en la industria del crudo?"*), donde el modelo puede aprovechar la estructura sintáctica y detectar sinónimos que los modelos clásicos omitirían por no compartir el mismo vocabulario exacto.